# Open-Weight Training Pipeline Lab

## Goal / Mục tiêu
Thực hành capacity plan và release gates mà không cần job nhiều GPU.

## Setup / Chuẩn bị
Các giả định được đặt ở một cell; thay đổi chúng để tạo hai kịch bản.

In [ ]:
PARAMS_B = 1.0
TOKENS_B = 0.2
GPUS = 1
PEAK_TFLOPS = 100.0
MFU = 0.35
FREE_DISK_GIB = 80.0
CHECKPOINT_GIB = 8.0


## Steps / Các bước
### 1. Capacity estimate

In [ ]:
flops = 6 * PARAMS_B * 1e9 * TOKENS_B * 1e9
days = flops / (GPUS * PEAK_TFLOPS * 1e12 * MFU) / 86400
required_disk = 3 * CHECKPOINT_GIB * 1.25
capacity = {'idealized_days': round(days, 2), 'required_disk_gib': required_disk, 'disk_gate': FREE_DISK_GIB >= required_disk}
print(capacity)


### 2. Governance gates

In [ ]:
sources = [
    {'source_id': 'vi-001', 'license': 'mit', 'pii_class': 'none', 'sha256': 'a' * 64},
    {'source_id': 'vi-002', 'license': 'unknown', 'pii_class': 'possible', 'sha256': 'b' * 64},
]
allowed = {'mit', 'apache-2.0', 'cc-by-4.0', 'public-domain'}
blocked = [row['source_id'] for row in sources if row['license'] not in allowed or row['pii_class'] not in {'none', 'reviewed'}]
print({'blocked_sources': blocked})


## Checks / Kiểm tra

In [ ]:
assert capacity['disk_gate']
assert blocked == ['vi-002']
assert all(len(row['sha256']) == 64 for row in sources)
print('PASS')


## Next Steps / Bước tiếp
Lập hai kịch bản GPU/token khác nhau, thêm safety/evaluation gates và viết go/no-go memo.